# Qwen2.5-1.5B Fine-Tuning on Kaggle: Electrical Engineering (Signals & Systems)

Ready-to-run Kaggle Notebook to fine-tune **Qwen2.5-1.5B-Instruct** on the 100% leak-free **Signals & Systems** dataset (1,200 curated golden samples, ~8 mins) and export **`qwen2.5-1.5b-instruct-q4_k_m.gguf`** directly to `/kaggle/working/` for your Jetson Orin Nano.

### Before Clicking "Run All" on Kaggle:
1. Right sidebar → **Session options** → **Accelerator**: select **`GPU T4 x2`** (recommended) or **`GPU P100`**.
2. Right sidebar → **Session options** → **Internet**: toggle **ON**.
3. Click **Run All** (total time: ~10–12 minutes end-to-end).
4. When finished, download `/kaggle/working/qwen2.5-1.5b-instruct-q4_k_m.gguf` (~1.0 GB) from the right sidebar **Output** tab (or via the link printed in Step 6).

### Step 1: Install Dependencies & Verify Kaggle GPU
Keeps Kaggle's pre-installed CUDA PyTorch untouched and installs only the LoRA / GGUF packages.

In [ ]:
# Pin to GPU 0 so Kaggle T4 x2 never hits DataParallel + QLoRA conflicts
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip uninstall -y torchao -q
!pip install -q -U "transformers>=4.45.0" "peft>=0.13.0" "datasets>=2.20.0" "accelerate>=0.34.0" "bitsandbytes>=0.44.0" sentencepiece protobuf "gguf>=0.10.0"

import torch
print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU detected! Enable GPU T4 x2 or GPU P100 in Kaggle Session options."
print("GPU:", torch.cuda.get_device_name(0), "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

### Step 2: Clone Repo & Verify Dataset (into `/tmp` to Keep `/kaggle/working` Clean)

In [ ]:
!rm -rf /tmp/MTP
!git clone --depth 1 https://github.com/aYush007-blip/MTP.git /tmp/MTP
!gzip -d -k -f /tmp/MTP/training/data/train_fast.jsonl.gz
!gzip -d -k -f /tmp/MTP/training/data/val_fast.jsonl.gz
!wc -l /tmp/MTP/training/data/train_fast.jsonl /tmp/MTP/training/data/val_fast.jsonl
!ls -lh /tmp/MTP/training/data/

### Step 3: Fine-Tune Qwen2.5-1.5B-Instruct (150 Steps, ~8 Mins Total on T4 GPU)
*Uses the 1,200 golden Signals & Systems samples (`train_fast`), native FP16 Tensor Cores, attention-only LoRA (`q,k,v,o`), and non-reentrant activation checkpointing (VRAM stays < 5 GB).*

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python /tmp/MTP/training/train_qwen_signals.py \
    --model-id "Qwen/Qwen2.5-1.5B-Instruct" \
    --data-dir "/tmp/MTP/training/data" \
    --output-dir "/tmp/output_qwen_signals" \
    --dataset fast \
    --target-modules attention \
    --epochs 1 \
    --batch-size 4 \
    --grad-accum 2 \
    --max-length 512 \
    --lr 2e-4


### Step 4: Quick Sanity Test on Fine-Tuned Merged Model
Verifies Signals & Systems spoken-to-LaTeX output BEFORE quantizing to GGUF.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_DIR = "/tmp/output_qwen_signals/merged_model"
tok = AutoTokenizer.from_pretrained(MERGED_DIR)
mdl = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.float16, device_map={"": 0})

SYSTEM_PROMPT = (
    "You are an expert Electrical Engineering lecture transcriber and mathematical typesetter "
    "specializing in Signals and Systems.\n"
    "Convert spoken mathematical formulas, signals, transforms, variables, and equations into "
    "clean inline LaTeX ($...$) while preserving all surrounding English lecture prose verbatim.\n"
    "Rules:\n"
    "- Continuous-time signals use parentheses: $x(t)$, $y(t)$, $h(t)$, $u(t)$, $\\delta(t)$, $x_2(t)$.\n"
    "- Discrete-time sequences use brackets: $x[n]$, $y[n]$, $h[n]$, $u[n]$, $\\delta[n]$, $x[2n]$, $x[n-k]$.\n"
    "- Transforms use uppercase: $X(s)$, $H(s)$, $X(z)$, $H(z)$, $X(j\\omega)$, $X(e^{j\\Omega})$, $X[k]$.\n"
    "- Use ONLY inline math ($...$). Never wrap ordinary conversational English in LaTeX.\n"
    "- Output ONLY the typeset lecture transcript with no labels or commentary."
)

tests = [
    "Now let us look at the second signal which is x 2 of t equals cos omega 0 t, and find the Laplace transform of e power minus 3 t u of t.",
    "By the initial value theorem, x of 0 plus equals limit as s goes to infinity of s times X of s, while for discrete sequences y of n equals x of 2 n.",
    "Good morning everyone. Please remember that tomorrow's tutorial will cover analog filter design."
]

for i, t in enumerate(tests, 1):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Spoken transcript:\n{t}"}
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=200, do_sample=False)
    ans = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"--- Test {i} ---\nIN : {t}\nOUT: {ans}\n")

del mdl
torch.cuda.empty_cache()

### Step 5: Convert & Quantize to `Q4_K_M` GGUF for Jetson Orin Nano
Builds `llama-quantize` in `/tmp/llama.cpp` and saves **only** the final `~1.0 GB` GGUF file into `/kaggle/working/`.

In [ ]:
!rm -rf /tmp/llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /tmp/llama.cpp
!cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF
!cmake --build /tmp/llama.cpp/build --target llama-quantize -j$(nproc)

# Convert merged HF model -> FP16 GGUF in /tmp
!python3 /tmp/llama.cpp/convert_hf_to_gguf.py /tmp/output_qwen_signals/merged_model --outfile /tmp/qwen2.5-1.5b-ee-fp16.gguf

# Quantize FP16 -> Q4_K_M directly into /kaggle/working/
quant_bin = !find /tmp/llama.cpp/build -type f -name llama-quantize -perm -111 | head -n 1
quant_bin = quant_bin[0]
print(f"Using quantizer binary: {quant_bin}")
!{quant_bin} /tmp/qwen2.5-1.5b-ee-fp16.gguf /kaggle/working/qwen2.5-1.5b-instruct-q4_k_m.gguf Q4_K_M

# Free temporary FP16 file
!rm -f /tmp/qwen2.5-1.5b-ee-fp16.gguf
!ls -lh /kaggle/working/qwen2.5-1.5b-instruct-q4_k_m.gguf

### Step 6: Download `qwen2.5-1.5b-instruct-q4_k_m.gguf` & Transfer to Jetson
Download from the link below or from Kaggle's right sidebar **Output → `/kaggle/working`**.

Then transfer to your Jetson Orin Nano:
```bash
scp qwen2.5-1.5b-instruct-q4_k_m.gguf ayush02@<JETSON_IP>:/home/ayush02/models/qwen2.5-1.5b-instruct-q4_k_m.gguf
```

In [ ]:
from IPython.display import FileLink
FileLink("qwen2.5-1.5b-instruct-q4_k_m.gguf")